# GLM-OCR bills of exchange — evaluation

Evaluates the best checkpoint on the **1,065 held-out test documents**, with results broken down by document origin.

**Protocol.** Greedy decoding, same prompt as training. A field is correct when the normalised prediction equals the
annotation (lower-case; spaces, hyphens, slashes and dots removed). A wrong non-empty value counts as one false positive
and one false negative. A document is *fully correct* when all seven fields are correct.

### Real documents (scanned bills of exchange)

| Documents | Fully correct | Avg CER | Avg WER |
|---|---|---|---|
| 39 | **97.4 %** | 0.1 % | 0.9 % |

### Full test set (real + augmented + synthetic)

| Origin | Documents | Fully correct |
|---|---|---|
| Real | 39 | 97.4 % |
| Synthetic | 246 | 77.2 % |
| Augmented (blur, rotation, noise, lighting) | 780 | 68.3 % |
| **All** | **1,065** | **71.5 %** |

| Field (all 1,065 documents) | Accuracy | F1 |
|---|---|---|
| numero_traite | 97.3 % | 97.4 % |
| montant | 96.8 % | 96.9 % |
| date_echeance | 96.3 % | 96.5 % |
| tireur | 96.2 % | 96.6 % |
| domiciliation | 91.2 % | 91.6 % |
| tire | 86.7 % | 86.8 % |
| rib (20 digits) | 85.0 % | 85.1 % |
| **Mean field accuracy** | **92.8 %** | |

The augmented test images are deliberately degraded, which explains the lower scores than on real scans.

> **Limitation.** The test split was also used to select the best checkpoint during training, so these scores may be
> slightly optimistic. Only 39 real documents are available for testing.

> **Environment.** Kaggle notebook, NVIDIA Tesla T4 (16 GB), Python 3.12. GLM-OCR requires the development
> version of `transformers` (installed in the first cell). After the first cell, **restart the kernel**.
>
> **Data.** The datasets are private (client documents) and are not distributed. Paths of the form
> `/kaggle/input/datasets/<KAGGLE_USER>/...` refer to private Kaggle datasets and must be adapted.

> **Anonymisation.** Outputs containing client or supplier data and document images were removed.
> Only aggregate metrics are kept.

## 1. Environment setup (restart the kernel afterwards)

In [ ]:
import os, sys, subprocess, shutil, site
from pathlib import Path

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["OMP_NUM_THREADS"]        = "1"

def pip(cmd):
    r = subprocess.run([sys.executable, "-m", "pip"] + cmd,
                       capture_output=True, text=True)
    if r.returncode != 0:
        print("ERR:", r.stderr[-300:])
    return r

pip(["uninstall", "-y", "transformers", "tokenizers",
     "huggingface-hub", "bitsandbytes", "peft", "trl", "accelerate"])

r = pip(["install", "-q", "--no-cache-dir",
         "git+https://github.com/huggingface/transformers.git"])
print(("OK" if r.returncode==0 else "ERR"), "transformers @ git")

for p in ["huggingface-hub==0.28.1", "accelerate==1.3.0",
          "peft==0.13.2", "trl==0.11.4"]:
    r = pip(["install", "-q", "--no-cache-dir", p])
    print(("OK" if r.returncode==0 else "ERR"), p)

pip(["uninstall", "-y", "bitsandbytes"])
print("✅ bitsandbytes supprimé")

for sp in site.getsitepackages():
    bnb_file = Path(sp) / "peft" / "tuners" / "lora" / "bnb.py"
    if bnb_file.exists():
        bnb_file.write_text(
            "# patched\n"
            "def dispatch_bnb_8bit(*a, **k): return None\n"
            "def dispatch_bnb_4bit(*a, **k): return None\n"
        )
        print("✅ peft/bnb.py patché")
        break

for sp in site.getsitepackages():
    tv_dir = Path(sp) / "torchvision"
    if not tv_dir.exists(): continue
    (tv_dir / "extension.py").write_text(
        "_HAS_OPS = False\n"
        "def _has_ops(): return False\n"
        "def _check_cuda_version(): pass\n"
        "def _load_library(n): return None\n"
        "def _assert_has_ops(): return None\n"
    )
    (tv_dir / "_meta_registrations.py").write_text("# patched\n")
    (tv_dir / "__init__.py").write_text(
        '__version__ = "0.0.0-patched"\n'
        "from .extension import _HAS_OPS\n"
        "try:\n    from . import transforms\nexcept: transforms = None\n"
        "try:\n    from . import io\nexcept: io = None\n"
    )
    print("✅ torchvision patché")
    break

print("\n⚠️  RESTART KERNEL maintenant !")

OK transformers @ git
OK huggingface-hub==0.28.1
OK accelerate==1.3.0
OK peft==0.13.2
OK trl==0.11.4
✅ bitsandbytes supprimé
✅ peft/bnb.py patché
✅ torchvision patché

⚠️  RESTART KERNEL maintenant !


## 2. Evaluation
Runs on two T4 GPUs (`cuda:0` / `cuda:1`, as loaded by `device_map='auto'`).

In [ ]:
import json, re, torch, statistics
from pathlib import Path
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
from peft import PeftModel
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')
from IPython.display import display, HTML

# ── Load best model ───────────────────────────────────────────
BEST_MODEL = Path('/kaggle/input/datasets/<KAGGLE_USER>/finalbest')
MODEL_NAME = 'zai-org/GLM-OCR'
MAX_SIZE   = 1024

print("⏳ Loading best model epoch 38...")
processor  = AutoProcessor.from_pretrained(
    MODEL_NAME, use_fast=False, trust_remote_code=True)
base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME, dtype=torch.float16,
    device_map='auto', trust_remote_code=True)
model = PeftModel.from_pretrained(base_model, str(BEST_MODEL))
model.eval()
print("✅ Model loaded")

# ── Load ALL test data ────────────────────────────────────────
BASE_DATA = Path('/kaggle/input/datasets/<KAGGLE_USER>/lettredechangeglm'
                 '/lettredechange_glm_zhai')
test_data = json.loads((BASE_DATA/'test_glmocr.json').read_text())

# ── Detect category from image path ──────────────────────────
def get_category(img_path):
    p = img_path.replace('\\','/').lower()
    if '/real/'      in p: return 'real'
    if '/augmented/' in p: return 'augmented'
    if '/generated/' in p: return 'generated'
    return 'unknown'

# Count by category
cat_counts = defaultdict(int)
for item in test_data:
    cat_counts[get_category(item['images'][0])] += 1
print(f"\n📊 Test set breakdown:")
for cat, n in sorted(cat_counts.items()):
    print(f"   {cat:<12} : {n} samples")
print(f"   {'TOTAL':<12} : {len(test_data)} samples")

ENTITIES = ['numero_traite','date_echeance','montant',
            'tireur','tire','rib','domiciliation']

# ── EXACT SAME PROMPT AS TRAINING ────────────────────────────
PROMPT = """Tu es un expert OCR spécialisé dans les lettres de change tunisiennes (كمبيالة).
Analyse l'image et extrais les 7 champs suivants en te guidant par leur POSITION sur le document :

1. numero_traite
   → Coin HAUT DROIT du document, après "Ordre de paiement L-CN" ou "أمر بدفع الكمبيالة عدد"
   → Suite de chiffres (10-12 chiffres), aussi visible dans le code-barres en BAS
   → Exemple : 012345678901

2. date_echeance
   → Zone HAUT GAUCHE, champ "Echéance / حلول الأجل"
   → C'est la date FUTURE (pas la date de création)
   → Format JJ/MM/AAAA
   → Exemple : 15/09/2026

3. montant
   → Zone HAUT DROIT, champ "Montant / البلغ"
   → Valeur numérique avec point ou virgule décimale
   → Ignorer "Montant en lettres"
   → Exemple : 2000.000

4. tireur
   → Zone MILIEU GAUCHE, champ "Tireur / الساحب"
   → Nom de la société ou personne qui émet la traite
   → Souvent en MAJUSCULES ou tampon
   → Exemple : SOCIETE EXEMPLE SARL

5. tire
   → Zone BAS CENTRE, champ "Nom et adresse du Tiré / اسم وعنوان المسحوب عليه"
   → Prendre UNIQUEMENT la première ligne (nom/société), ignorer l'adresse
   → Exemple : SOCIETE CLIENTE

6. rib
   → Zone MILIEU, champ "RIB ou RIP du Tiré / المعرف البنكي"
   → Code étab + Code Agence + N° Compte + Clé, sans espaces, chiffres uniquement
   → Exemple : 01234567890123456789

7. domiciliation
   → Zone BAS DROIT, champ "Domiciliation / اسم وعنوان الفرع المسحوب عليه"
   → Nom de la banque uniquement (BIAT, BNA, STB, Attijari, UIB, BTK...)
   → Exemple : BIAT

RÈGLES STRICTES :
- Répondre UNIQUEMENT avec un objet JSON valide, aucun texte avant ou après
- Si un champ est illisible → "illisible"
- Si un champ est absent → "N/A"
- Pour le RIB : chiffres uniquement, ignorer les lettres parasites

FORMAT :
{"numero_traite": "...", "date_echeance": "...", "montant": "...", "tireur": "...", "tire": "...", "rib": "...", "domiciliation": "..."}"""

print(f"\n✅ Prompt — {len(PROMPT)} chars (identical to training)")

# ── Helpers ───────────────────────────────────────────────────
def find_image(img_path):
    p = img_path.replace('\\','/')
    f = BASE_DATA / p
    if f.exists(): return f
    m = list(BASE_DATA.rglob(Path(p).name))
    return m[0] if m else None

def run_inference(img_path):
    image = Image.open(img_path).convert('RGB')
    w, h  = image.size
    if max(w,h) > MAX_SIZE:
        r = MAX_SIZE/max(w,h)
        image = image.resize((int(w*r),int(h*r)), Image.LANCZOS)
    msgs = [{"role":"user","content":[
        {"type":"image","image":image},
        {"type":"text","text":PROMPT}]}]
    inputs = processor.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors='pt',
        processor_kwargs={"return_mm_token_type_ids":True,
                          "truncation":False,"padding":False})
    inputs = {k:v.to('cuda:1') if isinstance(v,torch.Tensor) else v
              for k,v in inputs.items()}
    if 'pixel_values' in inputs:
        inputs['pixel_values'] = inputs['pixel_values'].to('cuda:0')
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=200,
                             do_sample=False)
    text = processor.decode(
        out[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True)
    try:
        m = re.search(r'\{.*?\}', text, re.DOTALL)
        return json.loads(m.group()) if m else {}
    except:
        return {}

def edit_distance(a, b):
    a, b = list(a), list(b)
    d = list(range(len(b)+1))
    for i, ca in enumerate(a):
        d2 = [i+1]
        for j, cb in enumerate(b):
            d2.append(min(d2[-1]+1, d[j+1]+1,
                          d[j]+(0 if ca==cb else 1)))
        d = d2
    return d[-1]

def cer(ref, hyp):
    ref = str(ref).lower().strip()
    hyp = str(hyp).lower().strip()
    if len(ref)==0: return 0.0 if len(hyp)==0 else 1.0
    return min(edit_distance(ref,hyp)/len(ref), 1.0)

def wer(ref, hyp):
    r = str(ref).lower().split()
    h = str(hyp).lower().split()
    if len(r)==0: return 0.0 if len(h)==0 else 1.0
    return min(edit_distance(r,h)/len(r), 1.0)

def normalize(v):
    return re.sub(r'[\s\-/.]','', str(v).lower().strip())

def is_exact(gt, pred):
    return normalize(gt) == normalize(pred)

def is_partial(gt, pred):
    g, p = normalize(gt), normalize(pred)
    return (g in p or p in g) and g != '' and p != ''

# ── Evaluation — ALL 1065 test samples ───────────────────────
N_EVAL = len(test_data)
print(f"\n⏳ Evaluating ALL {N_EVAL} test samples "
      f"(real + augmented + generated)...")
print("   This will take ~20-30 minutes on T4 GPU\n")

# Global stats
stats_all = defaultdict(lambda:{
    'exact':0,'partial':0,'cer':[],'wer':[],'total':0,
    'tp':0,'fp':0,'fn':0})

# Per-category stats
stats_cat = {
    'real':      defaultdict(lambda:{'exact':0,'total':0,'cer':[],'wer':[]}),
    'augmented': defaultdict(lambda:{'exact':0,'total':0,'cer':[],'wer':[]}),
    'generated': defaultdict(lambda:{'exact':0,'total':0,'cer':[],'wer':[]}),
}

all_correct  = 0
cat_correct  = defaultdict(int)
cat_total    = defaultdict(int)
skipped      = 0

for i, item in enumerate(test_data):
    if i % 50 == 0:
        elapsed_pct = i/N_EVAL*100
        print(f"   [{i:4d}/{N_EVAL}] {elapsed_pct:.0f}%",
              flush=True)

    img_path = find_image(item['images'][0])
    if img_path is None:
        skipped += 1; continue
    try:
        gt = json.loads(item['conversations'][1]['content'])
    except:
        skipped += 1; continue

    cat       = get_category(item['images'][0])
    pred      = run_inference(str(img_path))
    row_exact = True

    for ent in ENTITIES:
        gv = str(gt.get(ent, '')).strip()
        pv = str(pred.get(ent, '')).strip()
        ex = is_exact(gv, pv)
        pa = is_partial(gv, pv)
        c  = cer(gv, pv)
        w  = wer(gv, pv)

        # Global
        stats_all[ent]['exact']   += int(ex)
        stats_all[ent]['partial'] += int(pa and not ex)
        stats_all[ent]['cer'].append(c)
        stats_all[ent]['wer'].append(w)
        stats_all[ent]['total']   += 1
        if ex:
            stats_all[ent]['tp'] += 1
        elif pv.lower() not in ('','n/a','illisible'):
            stats_all[ent]['fp'] += 1
            stats_all[ent]['fn'] += 1
        else:
            stats_all[ent]['fn'] += 1

        # Per category
        if cat in stats_cat:
            stats_cat[cat][ent]['exact'] += int(ex)
            stats_cat[cat][ent]['total'] += 1
            stats_cat[cat][ent]['cer'].append(c)
            stats_cat[cat][ent]['wer'].append(w)

        if not ex:
            row_exact = False

    if row_exact:
        all_correct += 1
        cat_correct[cat] += 1
    cat_total[cat] += 1

evaluated  = N_EVAL - skipped
global_acc = all_correct/evaluated*100 if evaluated>0 else 0

print(f"\n✅ Evaluation complete — {evaluated} samples evaluated")

# ── HTML Report ───────────────────────────────────────────────
def bar_color(v):
    if v >= 90: return '#1a7a3a'
    if v >= 75: return '#7a6a00'
    if v >= 55: return '#a04000'
    return '#a00020'

# Main table rows
rows = ""
for ent in ENTITIES:
    s    = stats_all[ent]
    n    = s['total']
    acc  = s['exact']/n*100 if n else 0
    par  = (s['exact']+s['partial'])/n*100 if n else 0
    ac   = statistics.mean(s['cer'])*100 if s['cer'] else 0
    aw   = statistics.mean(s['wer'])*100 if s['wer'] else 0
    mc   = statistics.median(s['cer'])*100 if s['cer'] else 0
    tp   = s['tp']; fp = s['fp']; fn = s['fn']
    prec = tp/(tp+fp)*100 if (tp+fp)>0 else 0
    rec  = tp/(tp+fn)*100 if (tp+fn)>0 else 0
    f1   = 2*prec*rec/(prec+rec) if (prec+rec)>0 else 0
    bw   = int(acc/100*90)
    col  = bar_color(acc)
    rows += f"""
    <tr>
      <td style="font-weight:bold;color:#111">{ent}</td>
      <td>
        <div style="background:#e0e0e0;border-radius:3px;
                    width:90px;height:12px;display:inline-block;
                    vertical-align:middle">
          <div style="background:{col};width:{bw}px;height:12px;
                      border-radius:3px"></div>
        </div>
        <span style="color:{col};margin-left:6px;font-weight:bold">
          {acc:.1f}%
        </span>
      </td>
      <td style="color:#1a5a8a">{par:.1f}%</td>
      <td style="color:{'#1a7a3a' if ac<10 else '#a04000'}">{ac:.1f}%</td>
      <td style="color:{'#1a7a3a' if aw<20 else '#a04000'}">{aw:.1f}%</td>
      <td style="color:#555">{mc:.1f}%</td>
      <td style="color:{'#1a7a3a' if prec>=80 else '#a04000'}">{prec:.1f}%</td>
      <td style="color:{'#1a7a3a' if rec>=80 else '#a04000'}">{rec:.1f}%</td>
      <td style="color:{'#1a7a3a' if f1>=80 else '#a04000'};font-weight:bold">{f1:.1f}%</td>
    </tr>"""

# Category breakdown rows
cat_rows = ""
for cat in ['real','augmented','generated']:
    n   = cat_total.get(cat,0)
    acc = cat_correct.get(cat,0)/n*100 if n>0 else 0
    # avg CER/WER across all entities for this category
    all_cer = []
    all_wer = []
    for ent in ENTITIES:
        all_cer += stats_cat[cat][ent]['cer']
        all_wer += stats_cat[cat][ent]['wer']
    avg_cer = statistics.mean(all_cer)*100 if all_cer else 0
    avg_wer = statistics.mean(all_wer)*100 if all_wer else 0
    col = bar_color(acc)
    bw  = int(acc/100*90)
    icon = {'real':'🖼️','augmented':'🔄','generated':'🤖'}.get(cat,'📄')
    cat_rows += f"""
    <tr>
      <td style="font-weight:bold;color:#111">{icon} {cat}</td>
      <td style="color:#555">{n}</td>
      <td>
        <div style="background:#e0e0e0;border-radius:3px;
                    width:90px;height:12px;display:inline-block;
                    vertical-align:middle">
          <div style="background:{col};width:{bw}px;height:12px;
                      border-radius:3px"></div>
        </div>
        <span style="color:{col};margin-left:6px;font-weight:bold">
          {acc:.1f}%
        </span>
      </td>
      <td style="color:{'#1a7a3a' if avg_cer<10 else '#a04000'}">{avg_cer:.1f}%</td>
      <td style="color:{'#1a7a3a' if avg_wer<20 else '#a04000'}">{avg_wer:.1f}%</td>
    </tr>"""

html = f"""
<style>
  .ev {{ border-collapse:collapse; width:100%;
         font-family:Arial,sans-serif; font-size:13px;
         border:1px solid #ccc; }}
  .ev th {{ background:#2c5f9e; color:#ffffff;
            padding:9px 12px; text-align:left;
            border-bottom:2px solid #1a4a80; }}
  .ev td {{ padding:7px 12px; border-bottom:1px solid #e0e0e0;
            background:#ffffff; color:#222; }}
  .ev tr:nth-child(even) td {{ background:#f5f8ff; }}
  .ev tr:hover td {{ background:#e8f0ff; }}
  .cv {{ border-collapse:collapse; width:60%;
         font-family:Arial,sans-serif; font-size:13px;
         border:1px solid #ccc; margin-top:16px; }}
  .cv th {{ background:#1a6e3a; color:#fff;
            padding:8px 12px; text-align:left; }}
  .cv td {{ padding:7px 12px; border-bottom:1px solid #e0e0e0;
            background:#ffffff; color:#222; }}
  .cv tr:nth-child(even) td {{ background:#f0fff4; }}
  h3 {{ color:#2c5f9e; font-family:Arial,sans-serif; }}
  h4 {{ color:#1a6e3a; font-family:Arial,sans-serif; margin-top:20px; }}
  .kpi {{ background:#ffffff; border:2px solid #ddd;
          border-radius:8px; padding:14px 22px;
          text-align:center; min-width:120px; }}
  .kpi-val {{ font-size:24px; font-weight:bold;
              font-family:Arial,sans-serif; }}
  .kpi-lbl {{ color:#666; font-size:11px;
              font-family:Arial,sans-serif; margin-top:4px; }}
</style>

<h3>📊 Full Evaluation — Best Model Epoch 38
    &nbsp;|&nbsp; ALL {evaluated} test samples
    (real + augmented + generated)</h3>

<!-- KPI cards -->
<div style="display:flex;gap:14px;margin-bottom:16px;flex-wrap:wrap">
  <div class="kpi" style="border-color:#1a7a3a">
    <div class="kpi-val" style="color:#1a7a3a">{global_acc:.1f}%</div>
    <div class="kpi-lbl">Full Record Accuracy<br>(all 7 fields correct)</div>
  </div>
  <div class="kpi" style="border-color:#2c5f9e">
    <div class="kpi-val" style="color:#2c5f9e">{evaluated}</div>
    <div class="kpi-lbl">Total samples<br>({skipped} skipped)</div>
  </div>
  <div class="kpi" style="border-color:#555">
    <div class="kpi-val" style="color:#555">{cat_total.get('real',0)}</div>
    <div class="kpi-lbl">🖼️ Real samples</div>
  </div>
  <div class="kpi" style="border-color:#555">
    <div class="kpi-val" style="color:#555">{cat_total.get('augmented',0)}</div>
    <div class="kpi-lbl">🔄 Augmented samples</div>
  </div>
  <div class="kpi" style="border-color:#555">
    <div class="kpi-val" style="color:#555">{cat_total.get('generated',0)}</div>
    <div class="kpi-lbl">🤖 Generated samples</div>
  </div>
  <div class="kpi" style="border-color:#7a6a00">
    <div class="kpi-val" style="color:#7a6a00">0.0352</div>
    <div class="kpi-lbl">Best eval loss<br>(epoch 38)</div>
  </div>
</div>

<!-- Per-field table -->
<h4>📋 Metrics per Field — All 1065 samples</h4>
<table class="ev">
  <tr>
    <th>Field</th>
    <th>Exact Accuracy</th>
    <th>Partial Match ↑</th>
    <th>Avg CER ↓</th>
    <th>Avg WER ↓</th>
    <th>Median CER ↓</th>
    <th>Precision ↑</th>
    <th>Recall ↑</th>
    <th>F1 Score ↑</th>
  </tr>
  {rows}
</table>

<!-- Per-category table -->
<h4>🗂️ Full Record Accuracy by Category</h4>
<table class="cv">
  <tr>
    <th>Category</th>
    <th>Samples</th>
    <th>Full Record Accuracy</th>
    <th>Avg CER ↓</th>
    <th>Avg WER ↓</th>
  </tr>
  {cat_rows}
</table>

<div style="margin-top:12px;padding:10px 14px;background:#f5f8ff;
            border:1px solid #ddd;border-radius:6px;
            font-family:Arial,sans-serif;font-size:11px;color:#555">
  <b>Metrics:</b>
  CER = Character Error Rate &nbsp;|&nbsp;
  WER = Word Error Rate &nbsp;|&nbsp;
  Exact = normalized match (lowercase, strip spaces/hyphens/dots) &nbsp;|&nbsp;
  Partial = one value contains the other &nbsp;|&nbsp;
  F1 = harmonic mean of Precision &amp; Recall<br>
  <b>Prompt:</b> Exact same prompt used during training (French + Arabic).
  &nbsp;|&nbsp;
  <b>Model:</b> GLM-OCR + LoRA r=4, epoch 38, eval_loss=0.0352
</div>
"""
display(HTML(html))

print(f"\n{'='*55}")
print(f"FINAL RESULTS — {evaluated} samples")
print(f"{'='*55}")
print(f"  Full record accuracy : {global_acc:.1f}%")
for cat in ['real','augmented','generated']:
    n   = cat_total.get(cat,0)
    acc = cat_correct.get(cat,0)/n*100 if n>0 else 0
    print(f"  {cat:<12} accuracy : {acc:.1f}%  ({n} samples)")
print(f"{'='*55}")

⏳ Loading best model epoch 38...


Loading weights:   0%|          | 0/510 [00:00<?, ?it/s]

✅ Model loaded

📊 Test set breakdown:
   augmented    : 780 samples
   generated    : 246 samples
   real         : 39 samples
   TOTAL        : 1065 samples

✅ Prompt — 1866 chars (identical to training)

⏳ Evaluating ALL 1065 test samples (real + augmented + generated)...
   This will take ~20-30 minutes on T4 GPU

   [   0/1065] 0%
   [  50/1065] 5%
   [ 100/1065] 9%
   [ 150/1065] 14%
   [ 200/1065] 19%
   [ 250/1065] 23%
   [ 300/1065] 28%
   [ 350/1065] 33%
   [ 400/1065] 38%
   [ 450/1065] 42%
   [ 500/1065] 47%
   [ 550/1065] 52%
   [ 600/1065] 56%
   [ 650/1065] 61%
   [ 700/1065] 66%
   [ 750/1065] 70%
   [ 800/1065] 75%
   [ 850/1065] 80%
   [ 900/1065] 85%
   [ 950/1065] 89%
   [1000/1065] 94%
   [1050/1065] 99%

✅ Evaluation complete — 1065 samples evaluated



FINAL RESULTS — 1065 samples
  Full record accuracy : 71.5%
  real         accuracy : 97.4%  (39 samples)
  augmented    accuracy : 68.3%  (780 samples)
  generated    accuracy : 77.2%  (246 samples)
